# ConRFT in simulation (Colab)
Runs the ConRFT pipeline on the simulated Franka cube-lifting task: scripted demos, then Cal-ConRFT offline pretraining, then HIL-ConRFT online fine-tuning with a simulated human.

**Runtime:** choose *Runtime → Change runtime type → GPU* (T4 is enough). Run the cells in order.

**Input:** the `conrft-sim` folder, either copied to `MyDrive/conrft-sim` in Google Drive or uploaded as `conrft-sim.zip` when the cell asks.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the `conrft-sim` folder

In [ ]:
import os, shutil, zipfile
SRC = "/content/conrft-sim"
WORK = "/content/work"          # clones + venv live here
DRIVE_OUT = None                # set below if Drive is mounted; checkpoints are copied there at the end
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir("/content/drive/MyDrive/conrft-sim"):
        shutil.copytree("/content/drive/MyDrive/conrft-sim", SRC, dirs_exist_ok=True)
        DRIVE_OUT = "/content/drive/MyDrive/conrft-sim-runs"
except Exception as e:
    print("Drive not used:", e)
if not os.path.isdir(SRC):
    from google.colab import files
    up = files.upload()  # pick conrft-sim.zip
    zipfile.ZipFile(next(iter(up))).extractall("/content")
print(sorted(os.listdir(SRC)))

## 2. Install (about 5 to 10 minutes)
Creates a Python 3.10 environment with `jax[cuda11_pip]==0.4.20`, TensorFlow 2.15, Octo, ConRFT's `serl_launcher` and SERL's `franka_sim`.

In [ ]:
%%bash
set -e
apt-get -qq install -y libegl1 libgl1 libosmesa6 > /dev/null
bash /content/conrft-sim/setup.sh gpu /content/work 2>&1 | tail -n 20

## 3. Check the GPU and the Octo-small weights
These are the two things I could not test outside Colab. If `jax.devices()` shows only a CPU, or Octo fails to load, stop here and fix that first.

In [ ]:
%%bash
source /content/work/.venv/bin/activate
python -c "import jax; print('jax devices:', jax.devices())"
python - <<'PY'
from octo.model.octo_model import OctoModel
m = OctoModel.load_pretrained("hf://rail-berkeley/octo-small")
print("Octo-small loaded, token size", m.config["model"]["token_embedding_size"])
PY

## 4. Test the simulator and the scripted expert
Expect shapes `(2, 256, 256, 3)`, `(2, 128, 128, 3)`, `(2, 7)` and an expert success rate near 29/30.

In [ ]:
%%bash
source /content/work/.venv/bin/activate
cd /content/work/conrft/examples
MUJOCO_GL=egl python test_sim_env.py --episodes 5 --save_frames /content/frames.npz

In [ ]:
import numpy as np, matplotlib.pyplot as plt
f = np.load("/content/frames.npz")["frames"]
idx = np.linspace(0, len(f) - 1, 5).astype(int)
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for a, i in zip(ax, idx):
    a.imshow(f[i]); a.set_title(f"step {i}"); a.axis("off")

## 5. Record demos with the scripted expert
The paper uses 20 demos per task. Each demo is about 60 MB.

In [ ]:
%%bash
source /content/work/.venv/bin/activate
cd /content/work/conrft/examples
N_DEMOS=20 bash experiments/sim_pick_cube/run_record_demos.sh 2>&1 | grep -vE "pad_mask|Nothing will be masked" | tail -n 15
ls -lh demo_data/

## 6. Stage 1: Cal-ConRFT offline pretraining
Trains on the demos only with $\beta_Q=0.1$ (Q weight) and $\beta_{BC}=1.0$ (BC weight). The paper uses 20k steps; lower `PRETRAIN_STEPS` for a quicker first run.

In [ ]:
import glob, os
DEMO_PATH = sorted(glob.glob("/content/work/conrft/examples/demo_data/sim_pick_cube_*.pkl"))[-1]
CKPT = "/content/work/ckpt"
os.environ.update(DEMO_PATH=DEMO_PATH, CKPT=CKPT, PRETRAIN_STEPS="20000", WANDB_MODE="offline")
print(DEMO_PATH)

In [ ]:
%%bash
source /content/work/.venv/bin/activate
cd /content/work/conrft/examples
bash experiments/sim_pick_cube/run_learner_conrft_pretrain.sh 2>&1 | grep -vE "pad_mask|Nothing will be masked" | tail -n 15
ls $CKPT

## 7. Stage 2: HIL-ConRFT online fine-tuning
The learner switches to $\beta_Q=1.0$, $\beta_{BC}=0.1$ and samples half of each batch from the demos. The actor runs the policy in the simulator, and the scripted expert takes over in a fraction of episodes (`CONRFT_SIM_INTERVENE_PROB`, default 0.2), like the human with the SpaceMouse. Both run in the background; the next cell shows their progress.

In [ ]:
%%bash
source /content/work/.venv/bin/activate
cd /content/work/conrft/examples
nohup bash experiments/sim_pick_cube/run_learner_conrft.sh > /content/learner.log 2>&1 &
sleep 90   # let the learner load the checkpoint and open its server
CONRFT_SIM_INTERVENE_PROB=0.2 nohup bash experiments/sim_pick_cube/run_actor_conrft.sh > /content/actor.log 2>&1 &
echo started

In [ ]:
# Re-run this cell to check progress.
!grep -aE "last return|Traceback|Error" /content/actor.log | tail -n 5
!tail -c 600 /content/learner.log; echo
!ls /content/work/ckpt

In [ ]:
# Stop training when you are done.
!pkill -f "train_conrft_octo.py" ; sleep 2 ; ls /content/work/ckpt

## 8. Evaluate a checkpoint
Set `STEP` to one of the `checkpoint_<step>` folders above. Reports the success rate over 20 episodes with no interventions.

In [ ]:
%%bash
source /content/work/.venv/bin/activate
cd /content/work/conrft/examples
STEP=$(ls $CKPT | grep -oE "checkpoint_[0-9]+" | sed 's/checkpoint_//' | sort -n | tail -1)
CONRFT_SIM_INTERVENE_PROB=0 bash experiments/sim_pick_cube/run_actor_conrft.sh \
    --eval_checkpoint_step=$STEP --eval_n_trajs=20 2>&1 | grep -aiE "success|return|time" | tail -n 10

## 9. Save the checkpoints and logs to Drive

In [ ]:
import shutil, time
if DRIVE_OUT:
    dst = f"{DRIVE_OUT}/{time.strftime('%Y%m%d-%H%M')}"
    shutil.copytree(CKPT, f"{dst}/ckpt")
    for log in ["/content/learner.log", "/content/actor.log"]:
        shutil.copy(log, dst)
    print("saved to", dst)
else:
    print("Drive not mounted; download /content/work/ckpt manually.")